# Equity Duration × ECB Monetary Policy Shocks

This notebook studies how **firm-level equity duration** shapes stock price reactions to **ECB monetary policy announcements**, using a high-frequency event-study design.

The central hypothesis is that **equities with longer duration—i.e. firms whose value is concentrated in distant future cash flows—are more sensitive to unexpected changes in discount rates**. Monetary policy surprises provide a natural laboratory to test this prediction, as they induce abrupt shifts in interest rates and risk premia that are plausibly exogenous to individual firms.

A key contribution of this notebook is the introduction and empirical implementation of a **novel, net-payout-based equity duration measure**, which serves as the **primary duration concept** throughout the analysis.

---

## Research Question

Do **high-duration equities react more strongly** to ECB monetary policy surprises than **low-duration equities**, and does this sensitivity differ across **policy shocks** and **information shocks**?

More specifically, this notebook addresses three questions:

1. Does equity duration amplify firm-level stock price reactions to unexpected ECB policy shocks?
2. Is duration sensitivity driven primarily by **discount-rate shocks** rather than **information shocks**?
3. How does the explanatory power of a **net-payout-based equity duration** compare to standard DCF-based duration measures?

---

## Measuring Equity Duration

### Primary Duration Measure: Net-Payout-Based Equity Duration

The primary duration measure in this notebook is a **net-payout-based equity duration**, constructed from a forward-looking valuation identity that links **equity prices, expected payouts, and discount rates**.

Rather than discounting exogenously specified cash flows at an assumed cost of capital, this approach infers the **timing of equity cash flows directly from market valuations**. Expected net payouts—defined as dividends plus share repurchases minus equity issuances—are forecast using firm-level state variables, and the firm-specific discount rate is recovered endogenously such that the present value of expected payouts matches the observed equity valuation.

Formally, equity duration is defined as the **payout-weighted average maturity of expected net payouts**, using the same discount rate that rationalizes the firm’s market-to-book ratio. This ensures internal consistency between prices, expected cash flows, and discounting.

This measure has several advantages:

- It is **forward-looking** and disciplined by observed equity prices.
- It captures both **cash-flow timing** and **discount-rate exposure** in a model-consistent way.
- It is directly tied to **shareholder-relevant cash flows**, rather than accounting-based proxies.
- It avoids imposing auxiliary asset-pricing assumptions such as a fixed discount rate or a CAPM structure.

Because this duration measure is constructed **ex ante** and remains fixed within each event window, it is well suited for causal analysis of stock price reactions to monetary policy shocks.

---

### Benchmark Duration Measures

To assess robustness and disentangle timing versus discounting effects, I compare the net-payout-based duration to two commonly used benchmark measures:

- **`Duration_CAPM`**  
  A discounted-cash-flow (DCF) equity duration that applies a **firm-specific CAPM-implied cost of equity**.  
  This measure captures both cash-flow timing and heterogeneity in required returns driven by systematic risk.

- **`Duration_r125`**  
  A DCF-based equity duration computed using a **constant discount rate of 12.5% for all firms**.  
  By fixing the discount rate, this measure isolates **pure cash-flow timing**, abstracting from cross-sectional variation in discount rates.

All duration measures are standardized prior to estimation. Comparing their performance allows me to assess whether monetary policy transmission operates primarily through **cash-flow horizons**, **discount-rate heterogeneity**, or a combination of both.

---

## ECB Monetary Policy Shocks

The analysis focuses on ECB announcement days and exploits a high-frequency identification strategy that decomposes monetary policy surprises into two orthogonal components:

- **`ShockMP`**: a *pure monetary policy shock*, interpreted as an unexpected tightening or easing that primarily affects discount rates.
- **`ShockInfo`**: an *information shock*, capturing news about the economic outlook conveyed by the ECB that moves yields and equities in the same direction.

This decomposition is crucial, as **equity prices may respond differently to discount-rate news than to information about future cash flows**. Duration-based predictions apply most cleanly to the former.

---

## Empirical Design

For each ECB announcement date, I compute **firm-level abnormal returns (AR)** and estimate panel regressions of the form:

$$
AR_{i,t} = \beta_0 + \beta_1 Shock_t + \beta_2 D_i + \beta_3 (Shock_t \times D_i) + \Gamma'X_i + \varepsilon_{i,t},
$$

where $D_i$ denotes a standardized equity duration measure and $X_i$ includes firm characteristics such as size and market beta.

The coefficient of interest is the interaction term $ \beta_3 $, which captures whether **equity duration amplifies firms’ sensitivity to monetary policy shocks**. Standard errors are **clustered by event date**, reflecting the fact that shocks are common across firms within each ECB meeting.

In extended specifications, I replace $Shock_t$ with `ShockMP` and `ShockInfo` to separately identify **discount-rate** and **information** channels of monetary transmission.

---

## Interpretation

Under the duration hypothesis, **unexpected policy tightenings should depress the prices of high-duration equities more strongly**, as their value depends disproportionately on distant cash flows that are more sensitive to increases in discount rates. By contrast, information shocks may have weaker or ambiguous duration effects, depending on how news about future economic conditions affects long-run cash-flow expectations.

By combining a novel net-payout-based duration measure with high-frequency ECB shock identification, this notebook provides a clean empirical test of the role of equity duration in monetary policy transmission.

In [ ]:
# Core
import numpy as np
import pandas as pd
from pathlib import Path
# Stats / regressions
import statsmodels.formula.api as smf
# Plots
import matplotlib.pyplot as plt

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

DUR_FILE = TABLE_DIR / "final_results_duration.xlsx"
SHOCK_FILE = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## 2. Load firm-level equity duration and characteristics

I load the firm-level duration measures and keep **Duration (CAPM r)** as the baseline equity-duration proxy.


In [39]:
# Load durations
df_dur_panel = load_parquet("equity_duration_panel")  # columns: RIC | CompanyName | YEAR | Duration_*

# Harmonize column names
if "YEAR" not in df_dur_panel.columns and "year" in df_dur_panel.columns:
    df_dur_panel = df_dur_panel.rename(columns={"year": "YEAR"})

# Basic cleaning
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# Choose baseline duration measure for the main analysis
DUR_COL = "Duration_NP"     
df_dur_panel[DUR_COL] = pd.to_numeric(df_dur_panel[DUR_COL], errors="coerce")

display(df_dur_panel[["RIC", "CompanyName", "YEAR", DUR_COL]].head())

,RIC,CompanyName,YEAR,Duration_NP
0,AMUN.PA,Amundi SA,2016,37.068811
1,AMUN.PA,Amundi SA,2017,38.387304
2,AMUN.PA,Amundi SA,2018,41.678062
3,AMUN.PA,Amundi SA,2019,35.481584
4,AMUN.PA,Amundi SA,2020,34.137037


## 3. Load ECB shock series


### Economic Meaning and Sign Conventions

I use the Jarociński–Karadi (2020) decomposition of high-frequency ECB surprises into two orthogonal components:

- **`ShockMP` (Monetary Policy Shock):** the “pure” policy component.  
  A **positive** value corresponds to an unexpected **tightening** (higher-than-expected short rates).  
  **Prediction:** on average, equity prices fall → abnormal returns should be negative.

- **`ShockInfo` (Information Shock):** captures the information revealed by the central bank about the outlook.  
  A **positive** value is interpreted as **good news** about fundamentals (often associated with rising yields and rising equities).  
  **Prediction:** equity prices may rise, depending on the balance of improved cash-flow expectations vs discount-rate effects.

I run both a one-shock (MP only) specification and a two-shock specification to separately identify discount-rate vs information channels.

In [40]:
df_shk = pd.read_csv(SHOCK_FILE)

# ------------------------------------------------------------
# 1. Parse dates
# ------------------------------------------------------------
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some ECB shock dates could not be parsed."

# ------------------------------------------------------------
# 2. Select shock decomposition
# ------------------------------------------------------------
# Available decompositions:
#
#   - "median" : General decomposition (baseline)
#                -> both monetary policy and information shocks
#                   can occur jointly within an announcement
#
#   - "pm"     : Restrictive decomposition (robustness)
#                -> mutually exclusive shocks ("poor man's" sign restrictions)
#
# Baseline choice for the paper: "median"

SHOCK_VERSION = "median"   # <-- BASELINE (recommended)

if SHOCK_VERSION == "median":
    shock_map = {
        "MP_median": "ShockMP",
        "CBI_median": "ShockInfo"
    }
elif SHOCK_VERSION == "pm":
    shock_map = {
        "MP_pm": "ShockMP",
        "CBI_pm": "ShockInfo"
    }
else:
    raise ValueError("SHOCK_VERSION must be 'median' or 'pm'.")

missing = [c for c in shock_map if c not in df_shk.columns]
if missing:
    raise ValueError(f"Missing required shock columns: {missing}")

df_shk = df_shk.rename(columns=shock_map)

# ------------------------------------------------------------
# 3. Optional: keep policy factor (pc1) for diagnostics only
# ------------------------------------------------------------
if "pc1" in df_shk.columns:
    df_shk["PolicyFactor"] = pd.to_numeric(df_shk["pc1"], errors="coerce")

# ------------------------------------------------------------
# 4. Keep one observation per ECB announcement date
# ------------------------------------------------------------
df_shk = (
    df_shk
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Numeric coercion
# ------------------------------------------------------------
for c in ["ShockMP", "ShockInfo"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

# ------------------------------------------------------------
# 6. Consistency check
# ------------------------------------------------------------
# NOTE:
# - For the restrictive decomposition ("pm"), MP + Info == pc1 holds by construction
# - For the general decomposition ("median"), this equality does NOT need to hold
#   and deviations are expected and economically meaningful

if "PolicyFactor" in df_shk.columns and SHOCK_VERSION == "pm":
    check = df_shk["ShockMP"] + df_shk["ShockInfo"] - df_shk["PolicyFactor"]
    print("Max |MP + Info − pc1| (pm):", check.abs().max())

# ------------------------------------------------------------
# 7. Final sanity output
# ------------------------------------------------------------
print("Shock decomposition used:", SHOCK_VERSION)
print("Number of ECB events:", df_shk.shape[0])

display(df_shk[["date", "ShockMP", "ShockInfo"]].head())

Shock decomposition used: median
Number of ECB events: 312


,date,ShockMP,ShockInfo
0,1999-01-07,0.020578,-0.058123
1,1999-01-21,0.008569,-0.004988
2,1999-02-18,-0.005565,0.005565
3,1999-03-04,-0.003596,0.001670
4,1999-03-18,-0.002326,0.001568


## 4. Event Panel Shocks x Duratons

### Load daily returns and construct abnormal returns (AR)

My baseline dependent variable is the **abnormal return** on each firm-event date.  
(Depending on the earlier construction in the notebook, this is either a market-adjusted return or a model-based abnormal return.)

To avoid relying on external index data (e.g., STOXX), I compute the daily market return directly from my sample:

$$
mkt\_ret_t = \frac{1}{N_t}\sum_i r_{i,t}
$$

and define abnormal returns as:

$$
AR_{i,t} = r_{i,t} - mkt\_ret_t
$$

This is a standard equal-weighted benchmark in event-study applications.

### Build the event-day panel (firm × ECB event date)

I keep only returns observed on ECB event dates and merge:
- shocks by `date`
- duration and firm controls by `RIC`

This creates the core event-study panel used in the regressions.

### Event Windows
- **Day 0:** the ECB announcement day (baseline window).
- **[0,+1] window:** a robustness check that sums Day 0 and Day +1 abnormal returns to capture delayed price adjustment.

Interpretation: the shock occurs on Day 0, but markets may incorporate information with a short delay, so [0,+1] provides a more conservative reaction measure.

### Standardize equity duration

I standardize duration to mean 0 and standard deviation 1:

$$
D^{std}_i = \frac{D_i - \bar{D}}{sd(D)}
$$

This makes interaction coefficients easy to interpret as the effect per **one standard deviation** increase in duration.



In [ ]:
# ============================================================
# Load daily returns (firm × day) and ensure AR exists
# ============================================================

df_ret = load_parquet("returns_daily")

# --- basic checks ---
assert "RIC" in df_ret.columns, "RIC missing in returns_daily"
assert "date" in df_ret.columns, "date missing in returns_daily"

df_ret["RIC"] = df_ret["RIC"].astype(str).str.strip()
df_ret["date"] = pd.to_datetime(df_ret["date"], errors="coerce")
assert df_ret["date"].notna().any(), "All dates are NaT after parsing."

print("returns_daily shape:", df_ret.shape)
print("Columns:", df_ret.columns.tolist()[:50])

# --- 1) Identify a daily return column if AR is missing ---
if "AR" not in df_ret.columns:
    # candidate names for raw daily returns
    candidates = [c for c in df_ret.columns if c.lower() in [
        "ret", "return", "returns", "daily_return", "tr.totalreturn", "tr.totalreturn1d",
        "tr.totalreturngross", "tr.pricepctchg", "pct_change", "r"
    ]]

    if len(candidates) == 0:
        # fallback heuristic: any numeric column with "ret" in name
        candidates = [c for c in df_ret.columns if ("ret" in c.lower() or "return" in c.lower())]

    if len(candidates) == 0:
        raise ValueError(
            "AR is missing and I could not infer a daily return column.\n"
            "Please tell me which column in returns_daily is the daily return."
        )

    RET_COL = candidates[0]
    print(f"[Info] AR not found. Using '{RET_COL}' as raw daily return column.")

    df_ret[RET_COL] = pd.to_numeric(df_ret[RET_COL], errors="coerce")

    # --- 2) Build a simple market-adjusted abnormal return: AR = ret - cross-sectional mean ret on that day ---
    # (This matches your earlier approach and is fine for cross-sectional heterogeneity.)
    mkt_ret = (
        df_ret.groupby("date")[RET_COL]
        .mean()
        .rename("mkt_ret")
        .reset_index()
    )

    df_ret = df_ret.merge(mkt_ret, on="date", how="left", validate="m:1")
    df_ret["AR"] = df_ret[RET_COL] - df_ret["mkt_ret"]

    print("[Info] Constructed AR = return - daily cross-sectional mean return.")

# --- final sanity checks ---
assert "AR" in df_ret.columns, "AR still missing after construction."
print("AR summary:")
display(df_ret["AR"].describe())
display(df_ret[["RIC", "date", "AR"]].head())


returns_daily shape: (3056104, 3)
Columns: ['date', 'RIC', 'ret']
[Info] AR not found. Using 'ret' as raw daily return column.
[Info] Constructed AR = return - daily cross-sectional mean return.
AR summary:


count      3056104.0
mean            -0.0
std         9.141328
min       -98.769514
25%        -0.928394
50%        -0.051051
75%         0.855764
max      8510.508424
Name: AR, dtype: Float64

,RIC,date,AR
0,1COVG.DE^L25,2015-10-07,-0.620827
1,1COVG.DE^L25,2015-10-08,-2.144274
2,1COVG.DE^L25,2015-10-09,0.348292
3,1COVG.DE^L25,2015-10-12,-0.190182
4,1COVG.DE^L25,2015-10-13,-0.414726


In [ ]:
# ============================================================
# Build firm × event-date panel and merge ECB shocks + durations
# + standardize each duration (new *_std columns)
# ============================================================

# --- 1) Event dates from shocks ---
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

event_dates = df_shk["date"].dropna().sort_values().unique()
print("Number of event dates:", len(event_dates))

# --- 2) Detect date column in df_ret ---
if "date" in df_ret.columns:
    DATE_COL = "date"
else:
    candidates = [c for c in df_ret.columns if c.lower() in ["day", "tradedate", "event_date", "datadate", "datetime"]]
    if len(candidates) == 0:
        raise ValueError(
            "Could not find a date column in df_ret. "
            "Rename your date column to 'date' or set DATE_COL manually."
        )
    DATE_COL = candidates[0]

print("Using df_ret date column:", DATE_COL)

# Parse dates in df_ret
df_ret[DATE_COL] = pd.to_datetime(df_ret[DATE_COL], errors="coerce")
assert df_ret[DATE_COL].notna().any(), "All df_ret dates are NaT after parsing."

# --- 3) Build event panel ---
df_evt = df_ret[df_ret[DATE_COL].isin(event_dates)].copy()
df_evt = df_evt.rename(columns={DATE_COL: "date"})
df_evt["RIC"] = df_evt["RIC"].astype(str).str.strip()

print("df_evt shape (firm × event dates):", df_evt.shape)
display(df_evt.head())

# --- 4) Merge shocks ---
df_evt = df_evt.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1"
)

miss_shk = df_evt[["ShockMP", "ShockInfo"]].isna().any(axis=1).mean()
print("Share of rows with missing shocks:", miss_shk)
assert miss_shk == 0, "Some event rows could not be matched to shocks."

# --- 5) Construct predetermined YEAR for duration merge (use year-end t-1) ---
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# --- 6) Merge durations (+ BETA_5Y, ME) ---
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# duration columns (keep your logic, but exclude keys)
duration_cols = [c for c in df_dur_panel.columns if c.startswith("D") and c not in ["DATE", "DAY", "RIC", "YEAR"]]
if len(duration_cols) == 0:
    raise ValueError("No duration columns found in df_dur_panel (expected columns starting with 'D' or 'Duration_').")

# extra firm-year controls to bring along (only if present)
extra_cols = [c for c in ["BETA_5Y", "ME"] if c in df_dur_panel.columns]

# numeric coercion
for c in duration_cols + extra_cols:
    df_dur_panel[c] = pd.to_numeric(df_dur_panel[c], errors="coerce")

df_evt = df_evt.merge(
    df_dur_panel[["RIC", "YEAR"] + duration_cols + extra_cols],
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1"
)

print("Merged duration cols:", duration_cols)
print("Merged extra cols:", extra_cols)
print("df_evt shape after merging shocks + durations + extras:", df_evt.shape)

# --- 6b) Standardize each duration into a separate *_std column (z-score) ---
# Standardization is done on the merged sample (df_evt), ignoring NaNs.

std_cols = []
for c in duration_cols:
    mu = df_evt[c].mean(skipna=True)
    sd = df_evt[c].std(skipna=True, ddof=0)  # ddof=0 -> population std; ddof=1 if sample std
    new_c = f"{c}_std"
    std_cols.append(new_c)

    if pd.isna(sd) or sd == 0:
        df_evt[new_c] = pd.NA
    else:
        df_evt[new_c] = (df_evt[c] - mu) / sd

print("Standardized duration columns added:", std_cols)

# --- 7) Diagnostics ---
print("\nMissing share by duration column:")
display(df_evt[duration_cols].isna().mean().sort_values())

print("\nMissing share by standardized duration column:")
display(df_evt[std_cols].isna().mean().sort_values())

print("\nPreview merged columns:")
display(df_evt[["RIC", "date", "YEAR", "ShockMP", "ShockInfo"] + duration_cols + std_cols].head(10))

Number of event dates: 312
Using df_ret date column: date
df_evt shape (firm × event dates): (136842, 5)


,date,RIC,ret,mkt_ret,AR
11,2015-10-22,1COVG.DE^L25,3.585657,1.34795,2.237708
41,2015-12-03,1COVG.DE^L25,-2.86533,-2.109469,-0.755861
72,2016-01-21,1COVG.DE^L25,3.97295,1.814211,2.158739
107,2016-03-10,1COVG.DE^L25,-0.812586,-0.921107,0.10852
135,2016-04-21,1COVG.DE^L25,1.04227,-0.120709,1.162978


Share of rows with missing shocks: 0.0
df_evt shape after merging shocks + durations: (136842, 14)
Standardized duration columns added: ['Duration_NP_std', 'Duration_NP_w_std', 'Duration_undiscounted_std', 'Duration_r125_std', 'Duration_CAPM_std', 'Duration_emp_2yOIS_std']

Missing share by duration column:


Duration_NP              0.317308
Duration_NP_w            0.317308
Duration_emp_2yOIS       0.362630
Duration_r125            0.419491
Duration_undiscounted    0.420310
Duration_CAPM            0.455642
dtype: float64


Missing share by standardized duration column:


Duration_NP_std              0.317308
Duration_NP_w_std            0.317308
Duration_emp_2yOIS_std       0.362630
Duration_r125_std            0.419491
Duration_undiscounted_std    0.420310
Duration_CAPM_std            0.455642
dtype: float64


Preview merged columns:


,RIC,date,YEAR,ShockMP,ShockInfo,Duration_NP,Duration_NP_w,Duration_undiscounted,Duration_r125,Duration_CAPM,Duration_emp_2yOIS,Duration_NP_std,Duration_NP_w_std,Duration_undiscounted_std,Duration_r125_std,Duration_CAPM_std,Duration_emp_2yOIS_std
0,1COVG.DE^L25,2015-10-22,2014,-0.074571,0.053691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1COVG.DE^L25,2015-12-03,2014,0.165226,-0.067779,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1COVG.DE^L25,2016-01-21,2015,-0.023532,0.002077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1COVG.DE^L25,2016-03-10,2015,0.002951,0.046962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1COVG.DE^L25,2016-04-21,2015,0.008748,-0.007144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1COVG.DE^L25,2016-06-02,2015,0.014169,-0.014699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1COVG.DE^L25,2016-07-21,2015,0.004206,0.021370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1COVG.DE^L25,2016-09-08,2015,0.029289,-0.015880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1COVG.DE^L25,2016-10-20,2015,-0.003432,0.002885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1COVG.DE^L25,2016-12-08,2015,-0.023695,0.024945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Why I Use Predetermined Duration (YEAR − 1)

To avoid **look-ahead bias**, I merge **firm-year duration measures from the previous year** to each ECB event.  
Concretely, for an ECB announcement in calendar year *t*, I use the equity duration measured at **year-end (12/31) of year *t−1***.

This ensures that duration is **known before the event** and cannot be mechanically affected by event-day returns.

**Implementation:**  
`YEAR = year(date) − 1` and merge on (`RIC`, `YEAR`).

## 7. Baseline regression: monetary policy shock × duration

### Core Hypotheses and Empirical Specification

This chapter studies heterogeneous stock price responses to ECB monetary policy announcements using a **two-shock framework** that explicitly separates **monetary policy shocks** from **central bank information shocks**. Throughout the analysis, I use the **general (median) decomposition** of Jarociński and Karadi (2020), which allows both types of shocks to occur jointly within a given announcement.

This specification is essential for analyzing equity duration, as it permits a clean separation between **discount-rate effects** and **cash-flow or information effects**.

---

#### Two-shock interaction regressions

The baseline empirical specification is:

$$
AR_{i,t}
=
\beta_1 \big( ShockMP_t \times D_{i,t-1} \big)
+
\beta_2 \big( ShockInfo_t \times D_{i,t-1} \big)
+
\Gamma' X_{i,t-1}
+
\tau_t
+
\varepsilon_{i,t},
$$

where $AR_{i,t}$ denotes the abnormal return of firm $i$ on ECB announcement date $t$,  
$D_{i,t-1}$ is a standardized, ex-ante equity duration measure,  
and $X_{i,t-1}$ includes firm-level controls such as size and market beta.  

Event fixed effects $\tau_t$ absorb the average market reaction to each announcement, so identification comes from **cross-sectional heterogeneity in duration within each event**.

---

#### Core hypotheses and expected signs

**Monetary policy shocks (`ShockMP`)**

A positive monetary policy shock corresponds to an unexpected tightening of policy and an increase in discount rates. Since high-duration equities derive a larger share of their value from distant cash flows, they should be more sensitive to such shocks.

- **Hypothesis 1 (Discount-rate channel):**
  $$
  \beta_1 < 0
  $$
  High-duration firms experience more negative abnormal returns following tightening surprises.

**Information shocks (`ShockInfo`)**

Information shocks capture news about the economic outlook conveyed by the ECB that affects expected cash flows rather than discount rates. The duration implication is therefore ambiguous ex ante.

- **Hypothesis 2 (Information / cash-flow channel):**  
  The sign of $\beta_2$ is an empirical question.
  - If information shocks primarily revise long-run growth expectations, long-duration firms may react more strongly ($\beta_2 > 0$).
  - If information shocks mainly affect near-term cash flows or cyclical firms, duration may play a limited role ($\beta_2 \approx 0$).

---

#### Equity duration measures

The primary duration measure is a **net-payout-based equity duration**, which infers the timing of shareholder cash flows from market valuations and expected net payouts. This measure is forward-looking, internally consistent with observed prices, and well suited for analyzing discount-rate sensitivity.

To assess robustness and disentangle underlying mechanisms, I compare results to two benchmark duration measures:

- **`Duration_CAPM`**: a DCF-based duration using firm-specific CAPM-implied discount rates, capturing both cash-flow timing and heterogeneity in required returns.
- **`Duration_r125`**: a DCF-based duration using a common discount rate of 12.5%, capturing primarily cash-flow timing.

All duration measures are standardized prior to estimation, and results are reported side-by-side across measures.

---

#### Inference and interpretation

Standard errors are **clustered by event date**, reflecting the fact that monetary policy and information shocks are common across firms within an ECB announcement.

The coefficients of interest are:
- `ShockMP × Duration`: heterogeneity in **discount-rate sensitivity** across firms.
- `ShockInfo × Duration`: heterogeneity in responses to **information and cash-flow news**.

Even if average market reactions are modest, statistically significant differences across duration measures indicate meaningful cross-sectional heterogeneity in monetary policy transmission to equity prices.

In [43]:
# ============================================================
# BASELINE REGRESSION
# Two-shock model with event fixed effects
# ============================================================

import pandas as pd
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# 0) Choose baseline standardized duration
# ------------------------------------------------------------
BASELINE_DUR_STD = "Duration_NP_std"   # primary duration measure

assert BASELINE_DUR_STD in df_evt.columns, (
    f"{BASELINE_DUR_STD} not found in df_evt."
)

# ------------------------------------------------------------
# 1) Build regression sample
# ------------------------------------------------------------
reg_cols = ["AR", "ShockMP", "ShockInfo", BASELINE_DUR_STD, "date"]

# Optional controls
if "ln_mktcap" in df_evt.columns:
    reg_cols.append("ln_mktcap")

HAS_BETA = 'Beta (5Y)' in df_evt.columns
if HAS_BETA:
    reg_cols.append("Beta (5Y)")

df_reg = df_evt[reg_cols].copy().dropna()

print("Baseline regression sample:", df_reg.shape)
print("Unique event dates:", df_reg["date"].nunique())

# ------------------------------------------------------------
# 2) Baseline formula
# ------------------------------------------------------------
# With event fixed effects, shock main effects are absorbed.
# Identification comes from cross-sectional heterogeneity within events.

formula = (
    f"AR ~ ShockMP:{BASELINE_DUR_STD} "
    f"+ ShockInfo:{BASELINE_DUR_STD} "
    f"+ C(date)"
)

if "ln_mktcap" in df_reg.columns:
    formula += " + ln_mktcap"

if HAS_BETA:
    formula += ' + Q("Beta (5Y)")'

print("Formula:", formula)

# ------------------------------------------------------------
# 3) Estimate with event-date clustered SEs
# ------------------------------------------------------------
m_baseline = smf.ols(formula, data=df_reg).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["date"]}
)

# ------------------------------------------------------------
# 4) Display only economically relevant coefficients
# ------------------------------------------------------------
res = pd.DataFrame({
    "coef": m_baseline.params,
    "std_err": m_baseline.bse,
    "z": m_baseline.tvalues,
    "pvalue": m_baseline.pvalues
})

display(
    res.loc[
        res.index.str.contains(
            r"Shock(MP|Info):|ln_mktcap|Beta", regex=True
        )
    ]
)

Baseline regression sample: (93421, 5)
Unique event dates: 312
Formula: AR ~ ShockMP:Duration_NP_std + ShockInfo:Duration_NP_std + C(date)


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_49802/2800332298.py:76: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res.index.str.contains(


,coef,std_err,z,pvalue
ShockMP:Duration_NP_std,-0.690817,0.376701,-1.833862,0.066674
ShockInfo:Duration_NP_std,0.987990,0.449809,2.196464,0.028059


## 8. Robustness: [0, +1] event window (two-day cumulative abnormal returns)

As a robustness check, I extend the event window to include the trading day following the ECB announcement and compute two-day cumulative abnormal returns:

$$
AR^{0,1}_{i,t} = AR_{i,t} + AR_{i,t+1}.
$$

This specification captures potential **delayed market reactions** and **post-announcement information digestion**, which may be particularly relevant for information shocks conveyed during the press conference.

**Implementation details:**
- The day $t+1$ return corresponds to the **next trading day for the same firm**, constructed by shifting abnormal returns within each firm (`groupby("RIC")`).
- If the next-day return is unavailable (e.g., due to end-of-sample truncation or delistings), the cumulative return is set to missing.
- Observations with missing $AR^{0,1}_{i,t}$ are dropped explicitly prior to estimation to ensure a consistent event-level clustering of standard errors.

The two-day cumulative return is then analyzed using the same **two-shock regression framework with event fixed effects** as in the baseline specification. This ensures that identification continues to rely on **cross-sectional heterogeneity within each ECB announcement**, while allowing for a more flexible adjustment horizon of equity prices.

In [44]:
# ============================================================
# 8. [0,+1] abnormal return window
# - builds AR_01
# - merges shocks (date)
# - merges predetermined, time-varying durations (RIC x YEAR)
# - standardizes durations within df_evt_01 -> new *_std columns
# - runs BASELINE two-shock regression with event FE + clustered SEs
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# --- 0) Build AR[0,+1] at firm-day level ---
df_ret_sorted = df_ret.copy()
df_ret_sorted["date"] = pd.to_datetime(df_ret_sorted["date"], errors="coerce")
df_ret_sorted = df_ret_sorted.sort_values(["RIC", "date"])

df_ret_sorted["AR0"] = pd.to_numeric(df_ret_sorted["AR"], errors="coerce")
df_ret_sorted["AR1"] = df_ret_sorted.groupby("RIC")["AR0"].shift(-1)
df_ret_sorted["AR_01"] = df_ret_sorted["AR0"] + df_ret_sorted["AR1"]

# --- 1) Keep only ECB event dates (announcement day = day 0) ---
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
event_dates = df_shk["date"].dropna().unique()

df_evt_01 = df_ret_sorted[df_ret_sorted["date"].isin(event_dates)].copy()
df_evt_01["RIC"] = df_evt_01["RIC"].astype(str).str.strip()

print("Event-panel [0,+1] shape (before merges):", df_evt_01.shape)

# --- 2) Merge shocks (m:1) ---
df_evt_01 = df_evt_01.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1"
)

miss_shk = df_evt_01[["ShockMP", "ShockInfo"]].isna().any(axis=1).mean()
print("Share of rows with missing shocks:", miss_shk)
assert miss_shk == 0, "Some event rows could not be matched to shocks."

# --- 3) Predetermined YEAR and merge durations (m:1) ---
# Predetermine duration using information available before the event year
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

df_dur_panel = df_dur_panel.copy()
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# Identify duration columns
duration_cols = [c for c in df_dur_panel.columns if c.startswith("Duration_")]
if len(duration_cols) == 0:
    duration_cols = [c for c in df_dur_panel.columns if c.startswith("D") and c not in ["DATE", "DAY"]]

if len(duration_cols) == 0:
    raise ValueError("No duration columns found in df_dur_panel (expected 'Duration_*' or 'D*').")

# Ensure numeric
for c in duration_cols:
    df_dur_panel[c] = pd.to_numeric(df_dur_panel[c], errors="coerce")

df_evt_01 = df_evt_01.merge(
    df_dur_panel[["RIC", "YEAR"] + duration_cols],
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1"
)

# Quick check: share of missing durations
miss_dur = df_evt_01[duration_cols].isna().all(axis=1).mean()
print("Share of rows with all durations missing after merge:", miss_dur)

# --- 4) Controls ---
if "MarketCap" in df_evt_01.columns:
    df_evt_01["ln_mktcap"] = np.log(pd.to_numeric(df_evt_01["MarketCap"], errors="coerce"))

# --- 5) Standardize durations IN THIS PANEL (new *_std columns) ---
std_cols = []
for c in duration_cols:
    mu = df_evt_01[c].mean(skipna=True)
    sd = df_evt_01[c].std(skipna=True, ddof=0)  # population sd (ddof=0)
    new_c = f"{c}_std"
    std_cols.append(new_c)

    if pd.isna(sd) or sd == 0:
        df_evt_01[new_c] = pd.NA
    else:
        df_evt_01[new_c] = (df_evt_01[c] - mu) / sd

print("Standardized duration columns added (first 10):", std_cols[:10])

# --- 6) Regression-ready sample ---
# Baseline duration for the paper (primary): Net-payout duration
BASELINE_DUR_STD = "Duration_NP_std"  # switch if needed

if BASELINE_DUR_STD not in df_evt_01.columns:
    raise ValueError(f"{BASELINE_DUR_STD} not found. Available std cols include: {std_cols[:15]} ...")

reg_cols_01 = ["AR_01", "ShockMP", "ShockInfo", BASELINE_DUR_STD, "date"]

if "ln_mktcap" in df_evt_01.columns:
    reg_cols_01.append("ln_mktcap")

HAS_BETA = 'Beta (5Y)' in df_evt_01.columns
if HAS_BETA:
    reg_cols_01.append("Beta (5Y)")

df_reg_01 = df_evt_01[reg_cols_01].copy().dropna()

print("Regression sample [0,+1]:", df_reg_01.shape)
print("Unique event dates:", df_reg_01["date"].nunique())

# --- 7) BASELINE two-shock regression on AR[0,+1] with event fixed effects ---
# With C(date), Shock main effects are absorbed (shocks constant within event),
# so we include only the interactions + event FE.
formula_01 = (
    f'AR_01 ~ ShockMP:{BASELINE_DUR_STD} + ShockInfo:{BASELINE_DUR_STD} + C(date)'
)

if "ln_mktcap" in df_reg_01.columns:
    formula_01 += " + ln_mktcap"

if HAS_BETA:
    formula_01 += ' + Q("Beta (5Y)")'

print("Formula:", formula_01)

m2_01 = smf.ols(formula_01, data=df_reg_01).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg_01["date"]}
)

res_01 = pd.DataFrame({
    "coef": m2_01.params,
    "std_err": m2_01.bse,
    "z": m2_01.tvalues,
    "pvalue": m2_01.pvalues,
})

# Nur die ökonomisch relevanten Interaktionen
key_rows = res_01.loc[
    res_01.index.str.contains(r"Shock(MP|Info):", regex=True)
]

display(key_rows)

Event-panel [0,+1] shape (before merges): (136842, 8)
Share of rows with missing shocks: 0.0
Share of rows with all durations missing after merge: 0.31730755177503983
Standardized duration columns added (first 10): ['Duration_NP_std', 'Duration_NP_w_std', 'Duration_undiscounted_std', 'Duration_r125_std', 'Duration_CAPM_std', 'Duration_emp_2yOIS_std']
Regression sample [0,+1]: (93417, 5)
Unique event dates: 312
Formula: AR_01 ~ ShockMP:Duration_NP_std + ShockInfo:Duration_NP_std + C(date)


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_49802/1507909557.py:144: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res_01.index.str.contains(r"Shock(MP|Info):", regex=True)


,coef,std_err,z,pvalue
ShockMP:Duration_NP_std,-0.337017,0.488169,-0.690370,0.489962
ShockInfo:Duration_NP_std,0.999113,0.726563,1.375124,0.169093


## 9. Portfolio-split regressions (non-linearity)

To allow for potential **non-linear duration effects**, I complement the continuous interaction regressions with **portfolio-split specifications** that compare firms with very different equity durations.

### Portfolio construction

For each calendar year, firms are sorted into terciles based on their **predetermined equity duration**.  
I then retain only firms in the **lowest** and **highest** terciles and define a binary indicator:

- `HighDur_i = 1` if firm $i$ belongs to the **high-duration tercile**,
- `HighDur_i = 0` if firm $i$ belongs to the **low-duration tercile**.

The middle tercile is excluded to sharpen the contrast between short- and long-duration firms.

---

### Two-shock portfolio-split regression

Using the same event-study framework as in the baseline analysis, I estimate the following regression:

$$
AR_{i,t}
=
\beta_1 \big( ShockMP_t \times HighDur_i \big)
+
\beta_2 \big( ShockInfo_t \times HighDur_i \big)
+
\tau_t
+
\varepsilon_{i,t},
$$

where $AR_{i,t}$ denotes the abnormal return of firm $i$ on ECB announcement date $t$, and $\tau_t$ are event fixed effects.

Because shocks are constant within each event, the inclusion of $\tau_t$ absorbs the average market reaction to the announcement. Identification therefore comes from **within-event differences** between high- and low-duration firms.

---

### Interpretation

The interaction terms have a direct and intuitive interpretation:

- `ShockMP × HighDur` measures how much **more (or less)** high-duration firms react to **monetary policy shocks** relative to low-duration firms.
- `ShockInfo × HighDur` measures differential responses to **information shocks** conveyed by the ECB.

This specification provides a transparent test for **non-linearities** in duration effects. Even if continuous interaction coefficients are small or imprecisely estimated, economically meaningful differences between high- and low-duration firms may still emerge in the tails of the duration distribution.

The portfolio-split results therefore serve as a complementary robustness check to the continuous interaction regressions.

In [45]:
# ============================================================
# Portfolio split (robust): year-by-year QUINTILES (Q1 vs Q5)
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df_evt = df_evt.copy()
df_evt["date"] = pd.to_datetime(df_evt["date"], errors="coerce")

# predetermined year (consistent with duration merge: t-1)
if "YEAR" not in df_evt.columns:
    df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# --- Choose duration used for sorting (predetermined, raw preferred) ---
BASE_DUR = "Duration_NP"   # e.g., "Duration_NP", "Duration_CAPM", "Duration_r125"
assert BASE_DUR in df_evt.columns, f"{BASE_DUR} not found in df_evt."

def safe_qcut_bins(x: pd.Series, q: int = 5, min_n: int = 100) -> pd.Series:
    """
    Assign quantile bins within a group (year).
    Robust to ties: qcut on ranks to avoid duplicate bin-edge issues.
    Returns NA if insufficient observations.
    """
    x = pd.to_numeric(x, errors="coerce")
    ok = x.notna()
    if ok.sum() < min_n:
        return pd.Series(pd.NA, index=x.index, dtype="object")

    r = x[ok].rank(method="average")

    try:
        # labels: Q1..Qq
        labels = [f"Q{k}" for k in range(1, q + 1)]
        b = pd.qcut(r, q=q, labels=labels)
        out = pd.Series(pd.NA, index=x.index, dtype="object")
        out.loc[b.index] = b.astype("object")
        return out
    except Exception:
        return pd.Series(pd.NA, index=x.index, dtype="object")

# 1) Build quintile groups year-by-year (predetermined)
df_evt["Dur_bin"] = df_evt.groupby("YEAR")[BASE_DUR].transform(lambda s: safe_qcut_bins(s, q=5, min_n=100))

print("Dur_bin value counts (including NA):")
display(df_evt["Dur_bin"].value_counts(dropna=False))

# 2) Keep only Bottom (Q1) vs Top (Q5)
df_evt_q15 = df_evt[df_evt["Dur_bin"].isin(["Q1", "Q5"])].copy()
df_evt_q15["HighDur"] = (df_evt_q15["Dur_bin"] == "Q5").astype(int)

print("Q1/Q5 sample shape:", df_evt_q15.shape)
display(df_evt_q15["Dur_bin"].value_counts())

# 3) Regression sample
reg_cols = ["AR", "ShockMP", "ShockInfo", "HighDur", "date"]

if "ln_mktcap" in df_evt_q15.columns:
    reg_cols.append("ln_mktcap")

HAS_BETA = 'Beta (5Y)' in df_evt_q15.columns
if HAS_BETA:
    reg_cols.append("Beta (5Y)")

df_reg_ps = df_evt_q15[reg_cols].copy().dropna()

print("Regression sample (Q1 vs Q5):", df_reg_ps.shape)
print("Unique event dates:", df_reg_ps["date"].nunique())

# 4) Two-shock portfolio-split regression WITH event fixed effects
# (Shock main effects absorbed by C(date))
formula = "AR ~ ShockMP:HighDur + ShockInfo:HighDur + C(date)"

if "ln_mktcap" in df_reg_ps.columns:
    formula += " + ln_mktcap"

if HAS_BETA:
    formula += ' + Q("Beta (5Y)")'

print("Formula:", formula)

m_ps = smf.ols(formula, data=df_reg_ps).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg_ps["date"]}
)

# 5) Show only relevant coefficients
res = pd.DataFrame({
    "coef": m_ps.params,
    "std_err": m_ps.bse,
    "z": m_ps.tvalues,
    "pvalue": m_ps.pvalues
})

display(res.loc[res.index.str.contains(r"Shock(MP|Info):HighDur|ln_mktcap|Beta", regex=True)])

Dur_bin value counts (including NA):


Dur_bin
<NA>    43421
Q1      18830
Q2      18701
Q3      18678
Q4      18638
Q5      18574
Name: count, dtype: int64

Q1/Q5 sample shape: (37404, 22)


Dur_bin
Q1    18830
Q5    18574
Name: count, dtype: int64

Regression sample (Q1 vs Q5): (37404, 5)
Unique event dates: 312
Formula: AR ~ ShockMP:HighDur + ShockInfo:HighDur + C(date)


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_49802/3566507334.py:96: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  display(res.loc[res.index.str.contains(r"Shock(MP|Info):HighDur|ln_mktcap|Beta", regex=True)])


,coef,std_err,z,pvalue
ShockMP:HighDur,-1.885898,1.050380,-1.795444,0.072583
ShockInfo:HighDur,3.183557,1.387112,2.295098,0.021728
